# One score, many capabilities

### seed variance & how the scoring protocol reshapes the leaderboard

The companion view to the _one truth vs. a hydra_ puzzle. Instead of one
model on one problem, we use **every repeated run in the eval grid**:
`models.parquet` holds ~30 reruns (seeds) of each model on each benchmark,
all at **identical settings** (temp 0.7, same decoding, same 50 problems in
the same order). Stack the seeds and each `(model, benchmark)` cell becomes a
**30 seeds × 50 problems** correctness matrix.

Two things a single accuracy number hides fall straight out of that matrix.
All the parsing/metrics live in `consistency.py`.


## 1. The seed axis — per-call instability

Each heatmap is one `(model, benchmark)` cell. **Every row is one seed** (one
rerun); **every column is one problem**, sorted easiest→hardest by how often
it's solved. Green = solved, grey = failed. A single accuracy number is just
the green fraction of the whole grid — and two grids with the _same_ green
fraction can look completely different.

Read the columns: a solid-green block (always solved), a grey block (always
failed), and — the interesting part — a **mottled middle band** where the
_same_ problem flips from seed to seed. Two regimes show up across the grid.


### Case A — capable but unstable: the coin-toss band

These models clearly _can_ do the task (most problems are solved by some
seed), but a large fraction of problems are genuine per-call coin tosses. The
wide mottled middle is the instability a single run hides.


In [1]:
import consistency as C

C.show_heatmap('gpt-oss-120b',     'game24')        # 84% of problems flip
C.show_heatmap('gpt-4.1-nano',     'sonnetwriting')  # 92% flip
C.show_heatmap('claude-haiku-4.5', 'humaneval')      # 76% flip (partial-credit greens)

Notice the **mottled band dominates** — these grids are mostly _neither_ solid
green nor solid grey. The reported accuracy (~55–65%) is essentially the
average outcome of a coin flip taken 50 × 30 times. Run any one of these once
and the number you get is close to noise.


### Case B — stable: decided, almost no flipping

Same kind of grid, opposite texture. Here nearly every problem is _decided_ —
solved on (almost) every seed or failed on (almost) every seed. The columns
are clean blocks, the middle band is thin. A single run is a trustworthy
estimate of the model's ability.


In [2]:
C.show_heatmap('gemini-3-flash',       'game24')    # ~0% flip, near-solid green
C.show_heatmap('Qwen3-235B-Thinking',  'scibench')  # 90% of problems fully decided
C.show_heatmap('gpt-5-mini',           'humaneval') # decided, with partial-credit shading

**The contrast is the whole point.** Case A and Case B can report similar
headline accuracies, yet one number means _"reliably this good"_ and the
other means _"this good on average, but any single run is a gamble."_ The
heatmap is what the scalar throws away. Per-call instability — Case A — is
exactly what test-time scaling (sampling + voting/verification) pays to
average out.


## 2. The protocol axis — "how you score it" _is_ the capability

Take that exact same matrix and read out several numbers people all call
"accuracy":

- **single-run** — one run's score (averaged over seeds): what a normal eval reports
- **worst / best seed** — the unlucky/lucky run you'd get by chance
- **maj@N** — is each problem solved by the _majority_ of seeds?
- **pass@N** — is each problem solved by _at least one_ seed? (the ceiling)


In [3]:
C.show_protocols('game24')

In [4]:
C.show_table('game24')

model,single-run,worst seed,best seed,maj@N,pass@N,flippy,seed Δ
gemini-3-flash,96%,90%,100%,100%,100%,0%,10
gpt-5-nano,94%,88%,100%,98%,100%,12%,12
gpt-5-mini,86%,76%,96%,96%,100%,32%,20
gpt-oss-120b,58%,42%,74%,70%,100%,84%,32
claude-haiku-4.5,45%,32%,58%,40%,100%,76%,26
gpt-4.1-mini,25%,14%,34%,20%,84%,40%,20
gpt-4.1-nano,21%,12%,34%,10%,78%,48%,22
DeepSeek-R1,2%,0%,6%,0%,38%,0%,6
Qwen3-235B-Thinking,0%,0%,2%,0%,6%,0%,2
Llama-4-Maverick,0%,0%,0%,0%,0%,0%,0


**What to take away.**

- **One model, two stories.** `claude-haiku-4.5` reads as **45%** single-run but
  **100% pass@N** — every Game24 problem is solvable for it; the 45% is a
  sampling tax. `gpt-oss-120b` is the same: 58% vs 100%. Whether you call these
  models weak or strong is a choice of protocol, not a fact about the model.
- **Lucky-seed inflation.** The _seed Δ_ column (best − worst single run) reaches
  **32 points**. Reporting a single run is reporting a single coin flip — and
  cherry-picking the best seed is free points.
- **The grader can manufacture incompetence.** `Qwen3-235B-Thinking` and
  `DeepSeek-R1` show ~0% on Game24 — not because they can't do it, but because
  the strict `io` extraction fails on how thinking models format answers (the
  same extraction trap from the instability notebook). That 0% is the sharpest
  case of all: _the methodology, not the model, produced the number._

Swap in any cell — each benchmark has its own personality:

```python
C.show_heatmap('gpt-4.1-mini', 'game24')
C.show_protocols('hotpotqa')
C.show_table('humaneval')
```

For the raw per-cell numbers without HTML: `python consistency.py`.
